# FinClub Open Project 2026 — IV Surface Imputation

Goal : Predicts missing implied volatility values in a Nifty options dataset by reconstructing the IV surface.

Approach used : 
1- create an option surface

2- Fill missing values using multiple interpolation techniques

3- Validate different methods internally

4- Learn optimal ensemble weights 

5- Generate final predictions and submission file

In [1]:
# Import all the required libraries 

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings("ignore")

from scipy.interpolate import Akima1DInterpolator
from scipy.optimize import minimize
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Define input Dataset paths-
DATA_PATH = "dataset.csv"
SAMPLE_SUB_PATH = "sandbox_solution.csv"

# Define output file names-
OUT_PRIVATE_SAFE = "submission.csv"
OUT_FILLED = "filled_dataset.csv"
OUT_REPORT = "validation_report.csv"
OUT_WEIGHTS = "ensemble_weights.csv"

## 1. Load Data 

In [2]:
df = pd.read_csv(DATA_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

In [3]:
df.shape

(975, 30)

In [4]:
df.head()

,datetime,underlying_price,NIFTY27JAN2625200CE,NIFTY27JAN2625300CE,NIFTY27JAN2625400CE,NIFTY27JAN2625500CE,NIFTY27JAN2625600CE,NIFTY27JAN2625700CE,NIFTY27JAN2625800CE,NIFTY27JAN2625900CE,...,NIFTY27JAN2624200PE,NIFTY27JAN2624300PE,NIFTY27JAN2624400PE,NIFTY27JAN2624500PE,NIFTY27JAN2624600PE,NIFTY27JAN2624700PE,NIFTY27JAN2624800PE,NIFTY27JAN2624900PE,NIFTY27JAN2625000PE,NIFTY27JAN2625100PE
0,07-01-2026 09:15,26111.65,0.12662,0.12330,0.11741,NaN,0.11005,0.10576,NaN,0.09724,...,0.15760,0.15240,0.14697,0.14105,0.13613,0.13085,0.12640,0.12142,0.11631,0.11150
1,07-01-2026 09:20,26141.40,0.08632,NaN,NaN,0.11779,0.11197,0.11028,NaN,NaN,...,NaN,0.15420,0.14753,0.14274,0.13849,0.13282,NaN,0.12363,NaN,0.11353
2,07-01-2026 09:25,26139.35,0.09147,NaN,0.09514,0.09933,0.09599,0.09204,0.09216,0.08954,...,0.15927,NaN,0.14919,0.14245,0.13806,0.13242,0.12877,0.12349,0.11817,NaN
3,07-01-2026 09:30,26128.95,0.10860,0.10842,0.11150,0.12248,0.10715,0.11098,0.10345,NaN,...,0.15755,NaN,0.14691,0.14209,0.13721,0.13184,0.12722,0.12252,0.11729,0.11200
4,07-01-2026 09:35,26131.90,0.10462,0.10538,0.12459,0.12051,0.11225,0.11294,0.10544,NaN,...,0.15924,0.15334,0.14784,0.14230,NaN,0.13219,0.12733,0.12295,0.11707,NaN


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 975 entries, 0 to 974
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   datetime             975 non-null    str    
 1   underlying_price     975 non-null    float64
 2   NIFTY27JAN2625200CE  781 non-null    float64
 3   NIFTY27JAN2625300CE  791 non-null    float64
 4   NIFTY27JAN2625400CE  773 non-null    float64
 5   NIFTY27JAN2625500CE  794 non-null    float64
 6   NIFTY27JAN2625600CE  795 non-null    float64
 7   NIFTY27JAN2625700CE  774 non-null    float64
 8   NIFTY27JAN2625800CE  765 non-null    float64
 9   NIFTY27JAN2625900CE  793 non-null    float64
 10  NIFTY27JAN2626000CE  794 non-null    float64
 11  NIFTY27JAN2626100CE  792 non-null    float64
 12  NIFTY27JAN2626200CE  766 non-null    float64
 13  NIFTY27JAN2626300CE  788 non-null    float64
 14  NIFTY27JAN2626400CE  762 non-null    float64
 15  NIFTY27JAN2626500CE  791 non-null    float64
 16  N

In [6]:
df.describe()

,underlying_price,NIFTY27JAN2625200CE,NIFTY27JAN2625300CE,NIFTY27JAN2625400CE,NIFTY27JAN2625500CE,NIFTY27JAN2625600CE,NIFTY27JAN2625700CE,NIFTY27JAN2625800CE,NIFTY27JAN2625900CE,NIFTY27JAN2626000CE,...,NIFTY27JAN2624200PE,NIFTY27JAN2624300PE,NIFTY27JAN2624400PE,NIFTY27JAN2624500PE,NIFTY27JAN2624600PE,NIFTY27JAN2624700PE,NIFTY27JAN2624800PE,NIFTY27JAN2624900PE,NIFTY27JAN2625000PE,NIFTY27JAN2625100PE
count,975.000000,781.000000,791.000000,773.000000,794.000000,795.000000,774.000000,765.000000,793.000000,794.000000,...,758.000000,786.000000,773.000000,780.000000,777.000000,800.000000,766.000000,780.000000,780.000000,770.000000
mean,25564.155846,0.140259,0.138818,0.138240,0.142228,0.146016,0.152041,0.160795,0.166556,0.170306,...,0.230603,0.222443,0.207041,0.191752,0.179123,0.166007,0.154250,0.141069,0.136577,0.126154
std,317.642518,0.068745,0.075571,0.090462,0.113038,0.141846,0.170909,0.198059,0.222128,0.228303,...,0.266993,0.249447,0.222475,0.190352,0.165855,0.135587,0.111492,0.069741,0.071401,0.052776
min,24940.700000,0.016800,0.063170,0.074030,0.080590,0.081800,0.084060,0.085290,0.086000,0.084500,...,0.149040,0.142520,0.134770,0.127890,0.116180,0.107260,0.101750,0.096340,0.089750,0.081910
25%,25266.850000,0.111710,0.110250,0.107390,0.105467,0.103865,0.102455,0.100830,0.099050,0.097935,...,0.161523,0.155110,0.148590,0.142550,0.136040,0.130223,0.125075,0.119525,0.114673,0.109355
50%,25624.200000,0.122760,0.118660,0.114480,0.112460,0.109560,0.107620,0.106660,0.106850,0.107955,...,0.171725,0.164835,0.157020,0.149170,0.141580,0.135060,0.128905,0.123640,0.118800,0.113695
75%,25768.200000,0.135670,0.128930,0.124020,0.121012,0.118900,0.117575,0.120410,0.125320,0.135595,...,0.183098,0.174737,0.164270,0.156512,0.148150,0.140690,0.134568,0.128752,0.123983,0.118760
max,26183.800000,0.548960,0.577860,0.731570,1.139080,1.532590,1.916340,2.292520,2.662460,3.027050,...,4.211720,3.829270,3.445750,3.060780,2.673900,2.284440,1.891510,0.755340,1.088710,0.592490


In [7]:
df.isnull().sum()

datetime                 0
underlying_price         0
NIFTY27JAN2625200CE    194
NIFTY27JAN2625300CE    184
NIFTY27JAN2625400CE    202
NIFTY27JAN2625500CE    181
NIFTY27JAN2625600CE    180
NIFTY27JAN2625700CE    201
NIFTY27JAN2625800CE    210
NIFTY27JAN2625900CE    182
NIFTY27JAN2626000CE    181
NIFTY27JAN2626100CE    183
NIFTY27JAN2626200CE    209
NIFTY27JAN2626300CE    187
NIFTY27JAN2626400CE    213
NIFTY27JAN2626500CE    184
NIFTY27JAN2623800PE    203
NIFTY27JAN2623900PE    201
NIFTY27JAN2624000PE    209
NIFTY27JAN2624100PE    176
NIFTY27JAN2624200PE    217
NIFTY27JAN2624300PE    189
NIFTY27JAN2624400PE    202
NIFTY27JAN2624500PE    195
NIFTY27JAN2624600PE    198
NIFTY27JAN2624700PE    175
NIFTY27JAN2624800PE    209
NIFTY27JAN2624900PE    195
NIFTY27JAN2625000PE    195
NIFTY27JAN2625100PE    205
dtype: int64

In [8]:
sample_sub.head()

,id,value
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.143129
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.102447
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.102447
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.147267
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.147267


In [9]:
sample_sub.shape

(5460, 2)

In [10]:
# Parse correctly, but keep original datetime string for submission IDs.
df["datetime_parsed"] = pd.to_datetime(df["datetime"], dayfirst=True)
df = df.sort_values("datetime_parsed").reset_index(drop=True)

option_cols = [c for c in df.columns if c not in ["datetime", "underlying_price", "datetime_parsed"]]

In [11]:
def parse_option(col):
    m = re.match(r"NIFTY(\d{2}[A-Z]{3}\d{2})(\d+)(CE|PE)$", col)
    if m is None:
        raise ValueError(f"Could not parse option column: {col}")
    return m.group(1), int(m.group(2)), m.group(3)

In [12]:
# extract strike price , optionn type , expiry details inforamtion from the dataset
# create metadata table for all options
meta = pd.DataFrame([
    {"column": c, "expiry": parse_option(c)[0], "strike": parse_option(c)[1], "type": parse_option(c)[2]}
    for c in option_cols
])

col_index = {c: i for i, c in enumerate(option_cols)}

In [13]:
groups = {}
j_to_group = {}
j_to_local_pos = {}

#group options separately for - Call options (CE) and Put Options (PE)
for opt_type in ["CE", "PE"]:
    tmp = meta.loc[meta["type"].eq(opt_type)].sort_values("strike")
    idx_arr = np.array([col_index[c] for c in tmp["column"]])
    groups[opt_type] = {
        "columns": tmp["column"].tolist(),
        "idx": idx_arr,
        "strike": tmp["strike"].to_numpy(dtype=float),
    }
    for pos, j in enumerate(idx_arr):
        j_to_group[j] = opt_type
        j_to_local_pos[j] = pos


In [14]:
# convert da=ys remaining untill expiry which can be useful for time-based filling
Y0 = df[option_cols].to_numpy(dtype=float)
expiry_dt = pd.Timestamp("2026-01-27 15:30")
days_to_expiry = ((expiry_dt - df["datetime_parsed"]).dt.total_seconds() / 86400).to_numpy()

print("Rows:", len(df))
print("Option columns:", len(option_cols))
print("Missing cells to predict:", int(np.isnan(Y0).sum()))

Rows: 975
Option columns: 28
Missing cells to predict: 5460


In [15]:
Y0

array([[0.12662, 0.1233 , 0.11741, ..., 0.12142, 0.11631, 0.1115 ],
       [0.08632,     nan,     nan, ..., 0.12363,     nan, 0.11353],
       [0.09147,     nan, 0.09514, ..., 0.12349, 0.11817,     nan],
       ...,
       [0.14225, 0.42629, 0.6745 , ..., 0.75534,     nan, 0.23712],
       [0.0168 , 0.39816, 0.69406, ...,     nan, 0.68798, 0.3404 ],
       [0.03726, 0.35816, 0.73157, ...,     nan, 1.08871,     nan]],
      shape=(975, 28))

In [16]:
Y0.shape

(975, 28)

## 2. Base Interpolation & Smoothing Functions

 define interpolation and smoothing functions
 
 these methods will estimates missing IV values using neighbouring dtrikes and market structure.

In [18]:
# core interpolation function
def fit_1d(x_obs, y_obs, x_all, method="linear", degree=2):
    x_obs = np.asarray(x_obs, dtype=float)
    y_obs = np.asarray(y_obs, dtype=float)
    x_all = np.asarray(x_all, dtype=float)

    if len(y_obs) == 0:
        return np.full_like(x_all, np.nan, dtype=float)
    if len(y_obs) == 1:
        return np.full_like(x_all, y_obs[0], dtype=float)

    order = np.argsort(x_obs)
    x_obs = x_obs[order]
    y_obs = y_obs[order]

    try:
        
        # Linear interpolation 
        
        if method == "linear":
            pred = np.interp(x_all, x_obs, y_obs)
            left = x_all < x_obs[0]
            right = x_all > x_obs[-1]
            if left.any():
                slope = (y_obs[1] - y_obs[0]) / (x_obs[1] - x_obs[0])
                pred[left] = y_obs[0] + slope * (x_all[left] - x_obs[0])
            if right.any():
                slope = (y_obs[-1] - y_obs[-2]) / (x_obs[-1] - x_obs[-2])
                pred[right] = y_obs[-1] + slope * (x_all[right] - x_obs[-1])

        # Akima Interpolation
        
        elif method == "akima":
            pred = Akima1DInterpolator(x_obs, y_obs)(x_all)
            if np.isnan(pred).any():
                fallback = fit_1d(x_obs, y_obs, x_all, method="linear")
                pred = np.where(np.isnan(pred), fallback, pred)
        
        # Polynomial Fitting
        
        elif method == "poly":
            d = min(degree, len(y_obs) - 1)
            mu = x_obs.mean()
            sd = x_obs.std() if x_obs.std() > 0 else 1.0
            coef = np.polyfit((x_obs - mu) / sd, y_obs, d)
            pred = np.polyval(coef, (x_all - mu) / sd)

        else:
            raise ValueError(method)

    except Exception:
        pred = fit_1d(x_obs, y_obs, x_all, method="linear")

    return np.clip(pred, 0.005, 8.0)

In [19]:
# This helper function will fill missing IV valeus row-wise using strike wise interpolation
def row_fill_matrix(Y, method="linear", degree=2):
    out = Y.copy()
    for opt_type, g in groups.items():
        idx = g["idx"]
        x = g["strike"]
        for i in range(Y.shape[0]):
            y = Y[i, idx]
            observed = ~np.isnan(y)
            if observed.all() or not observed.any():
                continue
            pred = fit_1d(x[observed], y[observed], x, method=method, degree=degree)
            out[i, idx[~observed]] = pred[~observed]
    return np.clip(out, 0.005, 8.0)

In [20]:
# This helper function will fill missing values using historical information from previous timestamps.
def past_time_fill_matrix(Y, fallback_matrix=None):
    if fallback_matrix is None:
        fallback_matrix = row_fill_matrix(Y, method="linear")
    out = pd.DataFrame(Y).ffill().to_numpy(dtype=float, copy=True)
    still_missing = np.isnan(out)
    if still_missing.any():
        out[still_missing] = fallback_matrix[still_missing]
    return np.clip(out, 0.005, 8.0)

It handles Missing values , Edge Extrapolation  and Sparse observations

Returns estimated IV values across all strikes

## 3. V2 Strict Surface Model

Use Akima interpolation for interior strikes
Use conservative methods near edges
Preserve realistic IV surface shape

In [21]:
def v2_strict_matrix(Y):

    # this will gnerate multiple candidate surfaces - Linear , Akima , Polynomial , historical(time -based)
    linear = row_fill_matrix(Y, method="linear")
    akima  = row_fill_matrix(Y, method="akima")
    poly2  = row_fill_matrix(Y, method="poly", degree=2)
    past   = past_time_fill_matrix(Y, fallback_matrix=linear)

    out = Y.copy()

    
    # detect whether a missing point is - Interior strike , Edge strike
    for opt_type, g in groups.items():
        idx = g["idx"]
        y_group = Y[:, idx]
        observed = ~np.isnan(y_group)

        left_any = np.zeros_like(observed, dtype=bool)
        left_any[:, 1:] = np.cumsum(observed[:, :-1], axis=1) > 0

        right_any = np.zeros_like(observed, dtype=bool)
        right_any[:, :-1] = np.cumsum(observed[:, 1:][:, ::-1], axis=1)[:, ::-1] > 0


        
        is_interior = left_any & right_any
        missing = ~observed

        # if Interior strike is missing point - mostly trust Akima interpolation
        interior_missing = missing & is_interior

        #if Edge strike is missing point - use a safer blend of linear , polynomial , historical information
        edge_missing = missing & ~is_interior

        akima_sub  = akima[:, idx]
        
        linear_sub = linear[:, idx]
        poly2_sub  = poly2[:, idx]
        past_sub   = past[:, idx]
        out_sub    = out[:, idx]

        out_sub[interior_missing] = 0.8 * akima_sub[interior_missing] + 0.2 * linear_sub[interior_missing]
        
        out_sub[edge_missing]     = 0.8 * linear_sub[edge_missing] + 0.1 * poly2_sub[edge_missing] + 0.1 * past_sub[edge_missing]

        out[:, idx] = out_sub

    # finally clip IV values inot a valid range
    return np.clip(out, 0.005, 8.0)

## 4. Validation Masks & Group Labelling
Create validation masks

Artificially hide some known valuesso we can measure how well each method reconstructs missing IVs

In [22]:
# helper function -Creates fake missing values for validation

def make_holdout_masks(Y, n_folds=2, mode="pattern", frac=0.15, seed=42):
    rng = np.random.default_rng(seed)
    observed = ~np.isnan(Y)
    masks = []

    for _ in range(n_folds):
        holdout = np.zeros_like(observed, dtype=bool)

    # validation mode - random :  it will randomly remove observed values      
        if mode == "random":
            positions = np.argwhere(observed)
            k = int(len(positions) * frac)
            chosen = rng.choice(len(positions), size=k, replace=False)
            holdout[positions[chosen, 0], positions[chosen, 1]] = True


    #  validation mode - Pattern :  Mimic the actual missing-value structure present in the competition dataset       
        elif mode == "pattern":
            for i in range(Y.shape[0]):
                for opt_type, g in groups.items():
                    idx = g["idx"]
                    available = idx[observed[i, idx]]
                    actual_missing_count = np.isnan(Y[i, idx]).sum()
                    mask_count = min(int(actual_missing_count), max(0, len(available) // 3))
                    if mask_count > 0 and len(available) > 2:
                        chosen = rng.choice(available, size=mask_count, replace=False)
                        holdout[i, chosen] = True

    
     # validation mode - edge :  it will specifically test difficult edge strikes   
        elif mode == "edge":
            for i in range(Y.shape[0]):
                for opt_type, g in groups.items():
                    idx = g["idx"]
                    available_local_pos = np.where(observed[i, idx])[0]
                    if len(available_local_pos) <= 3:
                        continue
                    edge_positions = [p for p in available_local_pos if p in [0, 1, len(idx)-2, len(idx)-1]]
                    if edge_positions:
                        holdout[i, idx[rng.choice(edge_positions)]] = True
                    if rng.random() < 0.60:
                        interior_positions = [p for p in available_local_pos if p not in edge_positions]
                        if interior_positions:
                            holdout[i, idx[rng.choice(interior_positions)]] = True

        masks.append(holdout)

    return masks
# with help of these masks can estimate real world performance

In [23]:
def simplex_fit(P, y):
    k = P.shape[1]
    def objective(w):
        return np.mean((P @ w - y) ** 2)
    result = minimize(
        objective,
        np.ones(k) / k,
        method="SLSQP",
        bounds=[(0, 1)] * k,
        constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}],
        options={"maxiter": 1000, "ftol": 1e-12},
    )
    return result.x

In [24]:
def is_edge_position(Y, i, j):
    opt_type = j_to_group[j]
    idx = groups[opt_type]["idx"]
    local_pos = j_to_local_pos[j]
    observed = ~np.isnan(Y[i, idx])
    return not (observed[:local_pos].any() and observed[local_pos + 1:].any())


In [25]:
def group_label_for_cell(Y, i, j):
    edge = is_edge_position(Y, i, j)
    opt_type = j_to_group[j]
    idx = groups[opt_type]["idx"]
    n_observed_same_type = np.sum(~np.isnan(Y[i, idx]))
    near_expiry = days_to_expiry[i] <= 1.5
    low_observation = n_observed_same_type <= 8
    return int(edge) * 4 + int(near_expiry) * 2 + int(low_observation)

## 5. Run Internal Validation & Collect Candidate Predictions

In [26]:
# generate validation datasets
def collect_validation_predictions(Y):

    # it will train all candidate filling methods:Linear,Akima,polynomial Degree 2,Polynomial Degree 3,Historical Fill and V2 Strict Surface
    candidate_names = ["linear", "akima", "poly2", "poly3", "time_past", "v2_strict"]
    masks, modes = [], []

    for mode, seed in [("pattern", 101), ("random", 102), ("edge", 103)]:
        new_masks = make_holdout_masks(Y, n_folds=2, mode=mode, frac=0.15, seed=seed)
        masks.extend(new_masks)
        modes.extend([mode] * len(new_masks))

    records, prediction_rows, y_rows, group_rows = [], [], [], []

    for mask_id, (holdout, mode) in enumerate(zip(masks, modes)):
        Y_train = Y.copy()
        Y_train[holdout] = np.nan

        linear = row_fill_matrix(Y_train, method="linear")
        candidates = {
            "linear":    linear,
            "akima":     row_fill_matrix(Y_train, method="akima"),
            "poly2":     row_fill_matrix(Y_train, method="poly", degree=2),
            "poly3":     row_fill_matrix(Y_train, method="poly", degree=3),
            "time_past": past_time_fill_matrix(Y_train, fallback_matrix=linear),
            "v2_strict": v2_strict_matrix(Y_train),
        }

        for name, pred in candidates.items():
            records.append({
                "validation_mode": mode,
                "fold": mask_id,
                "method": name,
                "mse": mean_squared_error(Y[holdout], pred[holdout]),
                "mae": mean_absolute_error(Y[holdout], pred[holdout]),
                "n_holdout": int(holdout.sum()),
            })

        for i, j in np.argwhere(holdout):
            prediction_rows.append([candidates[name][i, j] for name in candidate_names])
            y_rows.append(Y[i, j])
            group_rows.append(group_label_for_cell(Y_train, i, j))

    return candidate_names, pd.DataFrame(records), np.asarray(prediction_rows), np.asarray(y_rows), np.asarray(group_rows)

In [27]:
candidate_names, validation_report, P, y_valid, group_labels = collect_validation_predictions(Y0)

In [28]:
# measure - MSE , MAE
validation_report.groupby(["validation_mode", "method"])[["mse", "mae"]].mean()

mse       mae
validation_mode method                       
edge            akima      0.000088  0.002763
                linear     0.000083  0.002869
                poly2      0.000161  0.004151
                poly3      0.000138  0.003237
                time_past  0.004089  0.009572
                v2_strict  0.000119  0.002837
pattern         akima      0.000087  0.002276
                linear     0.000086  0.002564
                poly2      0.000119  0.003436
                poly3      0.000110  0.002733
                time_past  0.002596  0.008860
                v2_strict  0.000090  0.002294
random          akima      0.000088  0.002202
                linear     0.000089  0.002457
                poly2      0.000110  0.003407
                poly3      0.000075  0.002384
                time_past  0.001968  0.008671
                v2_strict  0.000073  0.002170

## 6. Learn Ensemble Weights
Instead of choosing one method,combine several methods together using data-driven weights.

In [29]:
simple_group_labels = (group_labels >= 4).astype(int)

weights_simple = {}
for label in sorted(np.unique(simple_group_labels)):
    mask = simple_group_labels == label
    weights_simple[int(label)] = simplex_fit(P[mask], y_valid[mask])

weights_private_safe = {}
for label in sorted(np.unique(group_labels)):
    mask = group_labels == label
    weights_private_safe[int(label)] = simplex_fit(P[mask], y_valid[mask])

In [30]:
def make_weight_table(weights_dict, model_name):
    rows = []
    for label, weights in weights_dict.items():
        row = {"model": model_name, "group_label": int(label)}
        for name, w in zip(candidate_names, weights):
            row[name] = float(w)
        rows.append(row)
    return pd.DataFrame(rows)

In [31]:
weights_df = pd.concat([
    make_weight_table(weights_simple, "simple_edge_interior"),
    make_weight_table(weights_private_safe, "private_safe_adaptive"),
], ignore_index=True)


In [33]:
weights_df.head()

,model,group_label,linear,akima,poly2,poly3,time_past,v2_strict
0,simple_edge_interior,0,0.098656,0.106790,1.759843e-01,0.498629,0.014778,0.105163
1,simple_edge_interior,1,0.160142,0.160142,2.122817e-01,0.312385,0.005189,0.149860
2,private_safe_adaptive,0,0.189409,0.191195,1.892468e-01,0.194320,0.044991,0.190838
3,private_safe_adaptive,1,0.181265,0.195883,1.834595e-01,0.196152,0.050281,0.192960
4,private_safe_adaptive,2,0.076158,0.153812,1.936656e-17,0.614108,0.017642,0.138281


In [34]:
weights_df.shape

(10, 8)

## 7. Generate Final Filled IV Matrix

In [35]:
linear_full = row_fill_matrix(Y0, method="linear")
base_fills = {
    "linear":    linear_full,
    "akima":     row_fill_matrix(Y0, method="akima"),
    "poly2":     row_fill_matrix(Y0, method="poly", degree=2),
    "poly3":     row_fill_matrix(Y0, method="poly", degree=3),
    "time_past": past_time_fill_matrix(Y0, fallback_matrix=linear_full),
    "v2_strict": v2_strict_matrix(Y0),
}

Y_private_safe = Y0.copy()
missing_positions = np.argwhere(np.isnan(Y0))

In [36]:
for i, j in missing_positions:
    preds = np.asarray([base_fills[name][i, j] for name in candidate_names])
    adaptive_label = group_label_for_cell(Y0, i, j)
    Y_private_safe[i, j] = np.clip(preds @ weights_private_safe[adaptive_label], 0.005, 8.0)

In [37]:
filled_df = df.drop(columns=["datetime_parsed"]).copy()

filled_values = filled_df[option_cols].to_numpy(dtype=float, copy=True)

# Fill only originally missing positions
filled_values[np.isnan(Y0)] = Y_private_safe[np.isnan(Y0)]

filled_df[option_cols] = filled_values

filled_df.to_csv(OUT_FILLED, index=False)

print(f"Filled dataset saved -> {OUT_FILLED}")


Filled dataset saved -> filled_dataset.csv


## 8. Generate Submission csv file

In [38]:
# Generate Submission File


SEPARATOR = "||"

def generate_submission(original_df, filled_df, output_path):
    
    feature_cols = [c for c in original_df.columns if c != "datetime"]

    rows = []

    for col in feature_cols:

        missing_mask = original_df[col].isna()

        for idx in original_df.index[missing_mask]:

            dt = original_df.loc[idx, "datetime"]

            uid = f"{dt}{SEPARATOR}{col}"

            value = filled_df.loc[idx, col]

            rows.append({
                "id": uid,
                "value": float(value)
            })

    submission = pd.DataFrame(rows)

    submission = submission.sort_values("id").reset_index(drop=True)

    submission.to_csv(output_path, index=False)

    print(f"Submission saved -> {output_path}")
    print(f"Rows: {len(submission)}")

    return submission

In [39]:
# Create submission
sub_private = generate_submission(
    original_df=df.drop(columns=["datetime_parsed"]),
    filled_df=filled_df,
    output_path=OUT_PRIVATE_SAFE
)

Submission saved -> submission.csv
Rows: 5460


In [40]:
# Save Reports

validation_report.to_csv(OUT_REPORT, index=False)
weights_df.to_csv(OUT_WEIGHTS, index=False)

print("Validation Report Saved:", OUT_REPORT)
print("Weights Saved:", OUT_WEIGHTS)



Validation Report Saved: validation_report.csv
Weights Saved: ensemble_weights.csv


In [41]:
sub_private.head()

,id,value
0,07-01-2026 09:15||NIFTY27JAN2624100PE,0.163181
1,07-01-2026 09:15||NIFTY27JAN2625500CE,0.113583
2,07-01-2026 09:15||NIFTY27JAN2625800CE,0.101625
3,07-01-2026 09:20||NIFTY27JAN2624000PE,0.169836
4,07-01-2026 09:20||NIFTY27JAN2624200PE,0.159490
